In [1]:
# STEP 1: IMPORTS & OVERVIEW
# ----------------------------------------
# This script calculates NPV (Net Present Value) and IRR (Internal Rate of Return)
# for multiple retrofit measures in an EPC dataset.
#
# It:
# 1. Loads the dataset
# 2. Defines a discount factor (6%)
# 3. Calculates NPV using:
#       NPV = -CAP_COST + sum(annual_savings / (1 + r)^t)
# 4. Calculates IRR using numpy financial methods
# 5. Applies calculations to both FIXED and FLEX savings
# 6. Skips rows where required inputs are missing
# 7. Saves updated dataset back to file
#
# Designed for use in Jupyter / VSCode (Mac-compatible)

import pandas as pd
import numpy as np

In [2]:
# STEP 2: LOAD DATASET
# ----------------------------------------

file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_annual_data.csv"
df = pd.read_csv(file_path)

print("Dataset loaded. Shape:", df.shape)
df.head()

Dataset loaded. Shape: (117, 356)


,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,ORIENTATION,ORIENTATION_ESPR_ROT,...,PV_NPV_FLEX,BATTERY_NPV_FLEX,WINDOWS_IRR_FLEX,ROOF_IRR_FLEX,WALLS_IRR_FLEX,FAB_IRR_FLEX,HP_IRR_FLEX,SOLAR_THERMAL_IRR_FLEX,PV_IRR_FLEX,BATTERY_IRR_FLEX
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0557,-4.2233,100,170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.0528,-4.2206,180,90,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0555,-4.2237,100,170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.0584,-4.2248,150,120,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.0547,-4.2283,150,120,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# STEP 3: DEFINE PARAMETERS
# ----------------------------------------

discount_factor = 0.06

# Define all measures (prefixes used in column names)
measures = [
    "WINDOWS", "ROOF", "WALLS", "FAB",
    "HP", "SOLAR_THERMAL", "PV", "BATTERY"
]

In [4]:
# STEP 4 (UPDATED): IMPORT CORRECT IRR LIBRARY
# ----------------------------------------

# Install once if needed:
# !pip install numpy-financial

import numpy_financial as npf

In [5]:
# STEP 4: DEFINE FINANCIAL FUNCTIONS
# ----------------------------------------

def calculate_npv(cap_cost, annual_savings, lifetime, discount_rate):
    if pd.isna(cap_cost) or pd.isna(annual_savings) or pd.isna(lifetime):
        return np.nan
    
    try:
        lifetime = int(lifetime)
        cash_flows = [-cap_cost] + [annual_savings] * lifetime
        npv = sum(cf / ((1 + discount_rate) ** t) for t, cf in enumerate(cash_flows))
        return npv
    except:
        return np.nan


# STEP 5 (UPDATED): FIX IRR FUNCTION
# ----------------------------------------

def calculate_irr(cap_cost, annual_savings, lifetime):
    if pd.isna(cap_cost) or pd.isna(annual_savings) or pd.isna(lifetime):
        return np.nan
    
    try:
        lifetime = int(lifetime)
        
        # Avoid invalid cases
        if lifetime <= 0 or annual_savings == 0:
            return np.nan
        
        cash_flows = [-cap_cost] + [annual_savings] * lifetime
        
        irr = npf.irr(cash_flows)
        
        # Handle non-convergence or nonsense values
        if irr is None or np.isnan(irr) or np.isinf(irr):
            return np.nan
        
        return irr
    
    except:
        return np.nan

In [6]:
# STEP 5: CALCULATE FIXED SCENARIO (NPV + IRR)
# ----------------------------------------

for m in measures:
    cap_col = f"{m}_CAP_COST"
    life_col = f"{m}_LIFETIME"
    savings_col = f"{m}_COST_SAVINGS_FIXED"
    
    npv_col = f"{m}_NPV_FIXED"
    irr_col = f"{m}_IRR_FIXED"
    
    df[npv_col] = df.apply(
        lambda row: calculate_npv(
            row[cap_col],
            row[savings_col],
            row[life_col],
            discount_factor
        ),
        axis=1
    )
    
    df[irr_col] = df.apply(
        lambda row: calculate_irr(
            row[cap_col],
            row[savings_col],
            row[life_col]
        ),
        axis=1
    )

print("Fixed scenario calculations complete.")

Fixed scenario calculations complete.


In [7]:
# STEP 6: CALCULATE FLEX SCENARIO (NPV + IRR)
# ----------------------------------------

for m in measures:
    cap_col = f"{m}_CAP_COST"
    life_col = f"{m}_LIFETIME"
    savings_col = f"{m}_COST_SAVINGS_FLEX"
    
    npv_col = f"{m}_NPV_FLEX"
    irr_col = f"{m}_IRR_FLEX"
    
    df[npv_col] = df.apply(
        lambda row: calculate_npv(
            row[cap_col],
            row[savings_col],
            row[life_col],
            discount_factor
        ),
        axis=1
    )
    
    df[irr_col] = df.apply(
        lambda row: calculate_irr(
            row[cap_col],
            row[savings_col],
            row[life_col]
        ),
        axis=1
    )

print("Flex scenario calculations complete.")

Flex scenario calculations complete.


In [8]:
# STEP 7: SAVE UPDATED DATASET
# ----------------------------------------

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR.csv"
df.to_csv(output_path, index=False)

print("Updated dataset saved to:", output_path)

Updated dataset saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR.csv


In [9]:
# STEP 8: QUICK VALIDATION CHECK
# ----------------------------------------

# Check summary of newly created columns
npv_cols = [col for col in df.columns if "NPV" in col]
irr_cols = [col for col in df.columns if "IRR" in col]

print("NPV Summary:")
print(df[npv_cols].describe())

print("\nIRR Summary:")
print(df[irr_cols].describe())

NPV Summary:
                NPV  NPV_SCALED  WINDOWS_NPV_FIXED  ROOF_NPV_FIXED  \
count    117.000000  117.000000         117.000000      117.000000   
mean   -7813.500652    0.346529        1238.683055    16272.109044   
std    13108.433179    0.174284       28873.301970    33566.125846   
min   -33876.921200    0.000000      -38089.712936   -16128.646130   
25%   -14801.767610    0.253616      -15362.367517    -2219.519518   
50%    -6931.526078    0.358255       -9374.385901     -767.738211   
75%        0.000000    0.450414        6207.824756    24312.307477   
max    41335.928160    1.000000       91173.492149   113784.485206   

       WALLS_NPV_FIXED  FAB_NPV_FIXED  HP_NPV_FIXED  SOLAR_THERMAL_NPV_FIXED  \
count        65.000000      65.000000    117.000000               117.000000   
mean     -16142.330022  -29559.212929  -6703.222871               596.118084   
std        8704.365752   11021.036932  17908.953266              2743.718249   
min      -50846.032517  -69997.79300